In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, f1_score, accuracy_score, hamming_loss
from xgboost import XGBClassifier
from sklearn.multioutput import ClassifierChain
import mlflow
from pathlib import Path
from IPython.display import display, Markdown
import json

# ==========================================
# 1. SETUP PATH & MLFLOW
# ==========================================
root_path = Path.cwd().parent
mlflow.set_tracking_uri((root_path / "mlruns").as_uri())
mlflow.set_experiment("08_Performance_Evaluation")

# ✅ Load parameter hasil tuning (bukan hardcode)
params_path = root_path / "models" / "best_xgb_params.json"
with open(params_path, 'r') as f:
    xgb_params = json.load(f)
print(f"✓ Loaded tuned params: {xgb_params}")

print("⏳ 1. Memuat Data dan Model...")

# Load data training (fitur sudah diseleksi MFO)
train_df = pd.read_csv(root_path / "Data/processed/train_selected_features.csv")
test_df  = pd.read_csv(root_path / "Data/split/test_data.csv")

target_cols = ['risk_depression', 'risk_anxiety', 'risk_stress']

# ✅ DINAMIS: Fitur otomatis dari file MFO output
selected_features = [col for col in train_df.columns if col not in target_cols]

X_train = train_df[selected_features]
Y_train = train_df[target_cols].astype(int)

# ✅ Samakan kolom test dengan train (reindex)
X_test = test_df.reindex(columns=selected_features, fill_value=0)
Y_test = test_df[target_cols].astype(int)

print(f"   ✓ Fitur MFO: {len(selected_features)} fitur")
print(f"   ✓ Train: {X_train.shape}, Test: {X_test.shape}")

# ✅ TRAIN FRESH (tidak load pkl lama)
# xgb_params = {
#     'n_estimators': 300,
#     'learning_rate': 0.05,
#     'max_depth': 3,
#     'gamma': 0.2,
#     'colsample_bytree': 0.7,
#     'random_state': 42,
#     'eval_metric': 'logloss',
#     'n_jobs': -1
# }

base_xgb = XGBClassifier(**xgb_params)
model = ClassifierChain(base_xgb, order='random', random_state=42)
model.fit(X_train, Y_train)
print("   ✓ Model fresh dilatih")

with mlflow.start_run(run_name="Evaluate_Unseen_Test_Data"):
    # ==========================================
    # 2. PREDIKSI DATA UJIAN (Default threshold 0.5)
    # ==========================================
    print("⏳ 2. Memprediksi 3 Kondisi Mental secara bersamaan...")
    Y_pred = model.predict(X_test)
    
    # ==========================================
    # 3. MENGHITUNG METRIK MULTI-LABEL
    # ==========================================
    print("⏳ 3. Menghitung Metrik Evaluasi Khusus Multi-label...")
    
    exact_match = accuracy_score(Y_test, Y_pred)
    h_loss      = hamming_loss(Y_test, Y_pred)
    macro_f1    = f1_score(Y_test, Y_pred, average='macro')
    micro_f1    = f1_score(Y_test, Y_pred, average='micro')
    
    class_report_str = classification_report(
        Y_test, Y_pred, 
        target_names=['Depression', 'Anxiety', 'Stress']
    )
    
    # ==========================================
    # 4. LOGGING MLFLOW & OUTPUT
    # ==========================================
    mlflow.log_metric("test_exact_match",  exact_match)
    mlflow.log_metric("test_hamming_loss", h_loss)
    mlflow.log_metric("test_macro_f1",     macro_f1)
    mlflow.log_metric("test_micro_f1",     micro_f1)
    mlflow.log_param("num_features",       len(selected_features))
    mlflow.log_param("model_source",       "Fresh Training")
    mlflow.log_text(class_report_str, "classification_report.txt")
    
    summary = f"""
    ### 🎯 Tahap 08: Hasil Evaluasi Akhir (Unseen Test Data, Default Threshold 0.5)
    ---
    Model telah diuji menggunakan data testing murni sebanyak **{len(X_test):,} responden remaja** yang belum pernah dilihat sebelumnya.
    
    **Fitur yang digunakan:** {len(selected_features)} fitur (hasil seleksi MFO)
    
    **Metrik Evaluasi Multi-label Utama:**
    * **Macro F1-Score:** **{macro_f1:.4f}** (Rata-rata performa model pada ketiga kondisi mental).
    * **Micro F1-Score:** **{micro_f1:.4f}** (Performa keseluruhan berdasarkan agregat total prediksi benar).
    * **Hamming Loss:** **{h_loss:.4f}** (Tingkat kesalahan label hanya {h_loss*100:.2f}%).
    * **Exact Match Ratio:** **{exact_match:.4f}** (Persentase AI menebak kombinasi 3 label 100% sempurna).
    
    **Catatan:** Ini adalah evaluasi dengan threshold default (0.5).  
    Lihat NB 11 untuk evaluasi dengan threshold yang dioptimasi.
    
    **Laporan Klasifikasi Detail per Kondisi Mental:**
    ```text
    {class_report_str}
    ```
    ✅ *Evaluasi selesai. Metrik telah tersimpan di MLflow.*
    """
    display(Markdown(summary))
    print(class_report_str)


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


✓ Loaded tuned params: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.0833, 'gamma': 0, 'colsample_bytree': 0.5667, 'subsample': 0.8333, 'min_child_weight': 3, 'eval_metric': 'logloss', 'random_state': 42, 'n_jobs': -1}
⏳ 1. Memuat Data dan Model...
   ✓ Fitur MFO: 24 fitur
   ✓ Train: (8724, 24), Test: (5436, 24)
   ✓ Model fresh dilatih
⏳ 2. Memprediksi 3 Kondisi Mental secara bersamaan...
⏳ 3. Menghitung Metrik Evaluasi Khusus Multi-label...


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average


    ### 🎯 Tahap 08: Hasil Evaluasi Akhir (Unseen Test Data, Default Threshold 0.5)
    ---
    Model telah diuji menggunakan data testing murni sebanyak **5,436 responden remaja** yang belum pernah dilihat sebelumnya.
    
    **Fitur yang digunakan:** 24 fitur (hasil seleksi MFO)
    
    **Metrik Evaluasi Multi-label Utama:**
    * **Macro F1-Score:** **0.7813** (Rata-rata performa model pada ketiga kondisi mental).
    * **Micro F1-Score:** **0.7810** (Performa keseluruhan berdasarkan agregat total prediksi benar).
    * **Hamming Loss:** **0.2754** (Tingkat kesalahan label hanya 27.54%).
    * **Exact Match Ratio:** **0.5585** (Persentase AI menebak kombinasi 3 label 100% sempurna).
    
    **Catatan:** Ini adalah evaluasi dengan threshold default (0.5).  
    Lihat NB 11 untuk evaluasi dengan threshold yang dioptimasi.
    
    **Laporan Klasifikasi Detail per Kondisi Mental:**
    ```text
                  precision    recall  f1-score   support

  Depression       0.86      0.71      0.78      3842
     Anxiety       0.88      0.69      0.78      3995
      Stress       0.81      0.77      0.79      3304

   micro avg       0.85      0.72      0.78     11141
   macro avg       0.85      0.72      0.78     11141
weighted avg       0.86      0.72      0.78     11141
 samples avg       0.49      0.55      0.51     11141

    ```
    ✅ *Evaluasi selesai. Metrik telah tersimpan di MLflow.*
    

              precision    recall  f1-score   support

  Depression       0.86      0.71      0.78      3842
     Anxiety       0.88      0.69      0.78      3995
      Stress       0.81      0.77      0.79      3304

   micro avg       0.85      0.72      0.78     11141
   macro avg       0.85      0.72      0.78     11141
weighted avg       0.86      0.72      0.78     11141
 samples avg       0.49      0.55      0.51     11141

